# Faster R-CNN (ResNet-50 FPN v2) — evaluation

Scores the checkpoint written by [`faster_rcnn_train.ipynb`](faster_rcnn_train.ipynb) (`models/faster_rcnn/weights/best.pt`) on the test split.

Test images are only resized to 640×640 (no augmentation). Predictions, ground truth and per-image latency are collected in the
`EvaluationFramework` (see [`../example.ipynb`](../example.ipynb) for the metric definitions) and saved as JSON for the cross-model comparison.

In [ ]:
import time

import torch
from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2

from ppe_detection_evaluation import FASTER_RCNN, EvaluationFramework
from ppe_detection_evaluation.classes import DEFAULT_CLASS_NAMES
from ppe_detection_evaluation.dataset.core import check_dataset, list_images, load_eval_example, xyxy_to_cxcywh
from ppe_detection_evaluation.experiment import DATA_ROOT, IMAGE_SIZE, MODELS_DIR, RESULTS_DIR

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
NUM_CLASSES = len(DEFAULT_CLASS_NAMES) + 1  # + background at index 0
check_dataset(DATA_ROOT)  # the training notebook downloads the dataset

In [ ]:
# Same architecture as the training notebook. No pretrained weights are downloaded: the trained weights all come
# from the checkpoint.
model = fasterrcnn_resnet50_fpn_v2(
    weights=None,
    weights_backbone=None,
    num_classes=NUM_CLASSES,
    box_score_thresh=0.001,
)

checkpoint = MODELS_DIR / "faster_rcnn" / "weights" / "best.pt"
model.load_state_dict(torch.load(checkpoint, map_location=DEVICE))
model = model.to(DEVICE).eval()

In [ ]:
def sync():
    # Wait for queued GPU work, so the timer measures the forward pass rather than just its launch.
    if DEVICE.type == "cuda":
        torch.cuda.synchronize(DEVICE)
    elif DEVICE.type == "mps":
        torch.mps.synchronize()


fw = EvaluationFramework(FASTER_RCNN)
with torch.inference_mode():
    for path in list_images(DATA_ROOT, "test"):
        image, gt_boxes, gt_labels = load_eval_example(path)
        tensor = torch.from_numpy(image).permute(2, 0, 1).float().div(255).to(DEVICE)

        sync()
        start = time.perf_counter()
        out = model([tensor])[0]
        sync()
        latency_ms = (time.perf_counter() - start) * 1000

        fw.add(
            path.stem,
            out["boxes"].cpu().numpy(),
            out["scores"].cpu().numpy(),
            out["labels"].cpu().numpy(),
            xyxy_to_cxcywh(gt_boxes, IMAGE_SIZE, IMAGE_SIZE),
            gt_labels,
            latency_ms=latency_ms,
        )

results = fw.evaluate()
fw.save(RESULTS_DIR / "faster_rcnn_test.json")

In [ ]:
det = results["detection"]["groups"]
print(f"mAP@[.5:.95] {det['all']['mAP']:.4f}   mAP@.5 {det['all']['mAP50']:.4f}")
print(f"PPE mAP@.5 {det['ppe']['mAP50']:.4f}   small PPE mAP@.5 {results['detection']['by_size']['small']['groups']['ppe']['mAP50']:.4f}")
print("violation mean:", results["violation"]["mean"])
print("latency:", results["latency"])